## Original inputs


## Integration needs an explicit reference distribution

The visible qfunc composes two-asset loading, a five-bit exponential reference
and an indicator integration predicate. FlagQuantum lowers that reversible
composition and bounded Grover family. Host code derives normalized weights,
price offset, scale and strike-angle correction; the compiler does not discover
this integration method from a payoff function.

Try an admitted nonzero power and inspect full-state agreement before reading
only the indicator. Apply the declared affine rescaling before subtracting
strike; this differs from a direct-payoff scale. The 25-bit ceiling and all
original selectors remain. Sampling precision, finite-grid bias and private
dropped storage remain distinct from compiler cleanup and exact validation.


In [ ]:
from flagquantum import synth
from flagquantum.synth import Output
from flagquantum.synth import QBit
from flagquantum.synth import QNum
from flagquantum.synth import allocate
from flagquantum.synth import drop
from flagquantum.synth import phase
from flagquantum.synth import qfunc
from typing import Annotated

from examples.synthesis.applications.finance.rainbow_options.rainbow_options_integration_method.main import (
    pi,
    prepare,
    reflect,
)
from examples.synthesis.applications.finance.rainbow_options.rainbow_options_integration_method.overview.workflow import (
    Integrator,
    MAX_POWER,
    MAX_WIDTH,
)


@qfunc
def integration(
    k: int,
    probabilities: list[float],
    angles: list[float],
    lower_angle: float,
    indicator: Output[QBit],
) -> None:
    x: Annotated[QNum, 2] = QNum()
    y: Annotated[QNum, 2] = QNum()
    reference: Annotated[QNum, (5, False, 2)] = QNum()
    allocate(x)
    allocate(y)
    allocate(reference)
    allocate(indicator)
    prepare(probabilities, angles, lower_angle, x, y, reference, indicator)
    for iteration in range(k):
        phase(indicator, pi)
        reflect(probabilities, angles, lower_angle, x, y, reference, indicator)
    drop(x)
    drop(y)
    drop(reference)


def build_preview(model: Integrator, power: int) -> synth.Program:
    """Bind the original five-bit exponential reference and amplification power."""
    if type(power) is not int or not 0 <= power <= MAX_POWER:
        raise ValueError("Use a bounded Grover power in 0..4")
    return synth.synthesize(
        integration,
        classical_inputs={
            "k": power,
            "probabilities": model.asset_model.probabilities,
            "angles": model.angles,
            "lower_angle": model.lower_angle,
        },
        max_qubits=MAX_WIDTH,
        max_operations=100_000,
        memory_limit_bytes=4 * 1024**3,
    )


from IPython.display import Image, display

from examples.synthesis.applications.finance.rainbow_options.rainbow_options_direct_method.overview.model import (
    ALPHA,
    EPSILON,
    SOURCE_PRICE,
    STRIKE,
)
from examples.synthesis.applications.finance.rainbow_options.rainbow_options_integration_method.overview.main import (
    report,
    save_figure,
)
from examples.synthesis.applications.finance.rainbow_options.rainbow_options_integration_method.overview.workflow import (
    MAX_WIDTH,
    build_model,
    build_program,
    run_pricing,
)


## Regenerate the correlated asset model


In [ ]:
model = build_model()
asset = model.asset_model
print("Rounded coefficients:", asset.coefficients)


## Source strike and estimator settings


In [ ]:
print("Strike, epsilon and alpha:", STRIKE, EPSILON, ALPHA)


## Discretize both assets


In [ ]:
print("Original bin masses:", asset.probabilities)


## Check the payoff range


In [ ]:
print(
    "Reachable payoff range:",
    min(asset.payoff[index] for index in asset.selectors),
    max(asset.payoff[index] for index in asset.selectors),
)


## Inspect fixed-point precision


In [ ]:
print("Affine offset and threshold:", asset.offset, asset.threshold)


## Define the original rounded affine maximum


In [ ]:
print("All joint selectors:", asset.selectors)


## Check the selector bounds


In [ ]:
assert len(asset.selectors) == 16 and all(0 <= index < 32 for index in asset.selectors)


## Build the exponential integration reference


In [ ]:
print("Normalized reference weights:", model.weights)


## Derive the low-branch rotation


In [ ]:
print("Strike-price angle:", model.lower_angle)


## Synthesize the complete source preparation


In [ ]:
preparation = build_preview(model, 0)
print("Declared width bound and resources:", MAX_WIDTH, preparation.report["resources"])


## Set the local amplitude estimate tolerance


In [ ]:
print("Requested precision and confidence:", EPSILON, 1 - ALPHA)


## Execute and sample the native integration family


In [ ]:
result = run_pricing()
report(result)


## Scale by the original affine conversion


In [ ]:
print(
    "Scale and offset before strike subtraction:", model.price_scale, model.price_offset
)


## Compare with the independent finite model


In [ ]:
display(Image(filename=str(save_figure(result))))


## Check the source reference tolerance


In [ ]:
assert abs(result.price - SOURCE_PRICE) <= result.source_tolerance
